# ELBO Decomposition Explorer

Chapter 16: Variational Inference and Latent Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Decompose ELBO into expected log-likelihood and KL with a beta weight.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-16/elbo-decomposition/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

# The demo's defaults: observation x, encoder q(z|x) = N(mu, s^2), beta.
# Model: z ~ N(0, 1), x | z ~ N(z, 0.7^2).
x, mu, s, beta, sx = 1.2, 0.3, 0.9, 1.0, 0.7
prior = Normal(0.0, 1.0)
q = Normal(torch.tensor(mu, dtype=torch.float64), torch.tensor(s, dtype=torch.float64))

# ── Expected log-likelihood E_q[log N(x; z, sx^2)] in closed form ───────────
recon = -0.5 * math.log(2 * math.pi * sx ** 2) - ((x - mu) ** 2 + s ** 2) / (2 * sx ** 2)
kl = 0.5 * (s ** 2 + mu ** 2 - 1 - math.log(s ** 2))
print(f"E_q[log p(x|z)] = {recon:.6f}, KL(q || prior) = {kl:.6f}, ELBO = {recon - beta * kl:.6f}")

# Monte Carlo and torch's KL agree
z = q.sample((500000,))
print("Monte Carlo recon =", Normal(z, sx).log_prob(torch.tensor(x)).mean().item(),
      " torch KL =", kl_divergence(q, Normal(torch.tensor(0.0, dtype=torch.float64), torch.tensor(1.0, dtype=torch.float64))).item())

# ── The ELBO is a lower bound on log p(x), with gap KL(q || true posterior) ─
log_px = Normal(0.0, math.sqrt(1 + sx ** 2)).log_prob(torch.tensor(x)).item()   # x ~ N(0, 1 + sx^2)
post_mean, post_std = x / (1 + sx ** 2), math.sqrt(sx ** 2 / (1 + sx ** 2))
gap = kl_divergence(q, Normal(torch.tensor(post_mean, dtype=torch.float64), torch.tensor(post_std, dtype=torch.float64))).item()
print(f"\nlog p(x) = {log_px:.6f}")
print(f"ELBO + KL(q || p(z|x)) = {recon - kl + gap:.6f}   (identity: log p(x) = ELBO + gap)")
print(f"true posterior N({post_mean:.4f}, {post_std:.4f}^2); with q equal to it the gap is 0")

E_q[log p(x|z)] = -2.215325, KL(q || prior) = 0.055361, ELBO = -2.270685
Monte Carlo recon = -2.213736679331965  torch KL = 0.05536051565782628

log p(x) = -1.601548
ELBO + KL(q || p(z|x)) = -1.601548   (identity: log p(x) = ELBO + gap)
true posterior N(0.8054, 0.5735^2); with q equal to it the gap is 0
